In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-08

### Meteostat — Historical Weather Data

[Meteostat](https://dev.meteostat.net/) aggregates historical weather data from multiple national
meteorological agencies (NOAA, DWD, AEMET, etc.) into a unified Python API.
It is the **recommended alternative to OGIMET** for daily/hourly station data.

#### 🆚 Meteostat vs OGIMET — quick comparison

| Feature | Meteostat | OGIMET |
|---------|-----------|--------|
| Speed (55 years daily) | **< 30 s** | ~11 min+ (HTML scraping) |
| IP blocks / rate limits | No | Yes (cloud data centres blocked) |
| API stability | Official Python library | Fragile HTML scraping |
| Variables | Daily + Hourly | Full SYNOP (richer) |
| Resolution available | Daily / Hourly | Sub-daily raw SYNOP |

> **Use Meteostat** for long-period daily/hourly downloads — faster, more reliable, no login.
> **Use OGIMET** only when sub-daily raw SYNOP messages or cloud cover/visibility are required.

#### ⚖️ License

> ⚠️ Meteostat data is licensed under **CC BY-NC 4.0** — **non-commercial use only**.
> For commercial applications, contact the original data providers (NOAA, DWD, AEMET).

#### 📋 Available variables

| Column | Description | Unit |
|--------|-------------|------|
| `prcp` / `precipitation_mm` | Daily precipitation | mm |
| `temp` / `tmed_celsius` | Daily mean temperature | °C |
| `tmin` / `tmin_celsius` | Daily minimum temperature | °C |
| `tmax` / `tmax_celsius` | Daily maximum temperature | °C |
| `rhum` | Relative humidity (daily mean) | % |
| `wspd` | Mean wind speed | km/h |
| `wpgt` | Wind gust peak | km/h |
| `pres` | Sea-level pressure | hPa |
| `snow` | Snow depth | mm |
| `tsun` | Sunshine duration | min/day |

Not all stations have all variables. Use the **scoring step (Section 4)** to rank stations
by coverage before committing to a download.

#### 🗺️ Workflow overview

| Step | Section | Action |
|------|---------|--------|
| 1 | Configuration | Set coordinates, radius, period |
| 2 | Interactive map | Browse stations visually |
| 3 | Programmatic search | Find nearest stations to a point |
| 4 | Scoring | Rank stations by variable coverage |
| 5 | Download single | Download a specific station by WMO ID |
| 6 | Auto-select best | Automatic station selection + download |
| 7 | Quality analysis | Coverage and range checks |
| 8–10 | Visualisation + utils | Time series, annual cycle, WMO lookup |


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from pyhydra.data_sources.rainfall.meteostat import DEFAULT_METEOSTAT_OUTPUT_DIR
from pyhydra.data_sources.rainfall import (
    MeteostatDownloader,
    MeteostatCSVLoader,
    get_meteostat_stations_nearby,
    download_meteostat_series,
    score_meteostat_stations,
    select_best_meteostat_station,
    download_best_meteostat_series,
    find_station_by_wmo,
)

warnings.filterwarnings('ignore')
try:
    from IPython.display import display
except ImportError:
    display = print
print('pyhydra.data_sources.rainfall.meteostat loaded ✓')

pyhydra.data_sources.rainfall.meteostat loaded ✓


---
## 1. Configuration

Key parameters to adapt to your study:

| Parameter | Description |
|-----------|-------------|
| `LAT`, `LON` | Centre of your catchment or point of interest |
| `RADIUS_M` | Search radius for nearby stations (metres) |
| `START_DATE` / `END_DATE` | Desired record period |
| `FREQUENCY` | `'daily'` or `'hourly'` |
| `STATION_ID` | WMO code of a known station (override auto-selection) |

`RUN_DOWNLOADS` and `RUN_SMOKE` control behaviour in automated pipelines:
- **Smoke mode** (`RUN_SMOKE=1`, default): downloads only 1 month to validate the API path quickly
- **Full mode** (`RUN_DOWNLOADS=1`): downloads the full period — may take a few minutes


In [ ]:
# === CONFIGURATION ===
RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
RUN_SMOKE = os.getenv('HYDRA_DOWNLOAD_SMOKE', '1') == '1'

OUTPUT_DIR  = str(DEFAULT_METEOSTAT_OUTPUT_DIR) + '/'
START_DATE  = '1970-01-01'
END_DATE    = '2024-12-31'
FREQUENCY   = 'daily'

# Smoke mode keeps the release build fast while still validating the client/API path.
DOWNLOAD_START_DATE = '2020-01-01' if RUN_SMOKE and not RUN_DOWNLOADS else START_DATE
DOWNLOAD_END_DATE   = '2020-01-31' if RUN_SMOKE and not RUN_DOWNLOADS else END_DATE
SEARCH_LIMIT = 5 if RUN_SMOKE and not RUN_DOWNLOADS else 30

LAT, LON    = 43.259, -4.053
RADIUS_M    = 150_000
STATION_ID   = '08021'
STATION_NAME = 'Santander / Parayas'

df = None
nearby = pd.DataFrame()
quality_df = None
ranking = pd.DataFrame()

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output dir : {OUTPUT_DIR}')
print(f'Period     : {DOWNLOAD_START_DATE} → {DOWNLOAD_END_DATE}')
print(f'Frequency  : {FREQUENCY}')
print(f'Center     : ({LAT}, {LON})  radius = {RADIUS_M/1000:.0f} km')
print(f'Run smoke  : {RUN_SMOKE} | Run full downloads: {RUN_DOWNLOADS}')

Output dir : /workspace/data/meteostat/
Period     : 2020-01-01 → 2020-01-31
Frequency  : daily
Center     : (43.259, -4.053)  radius = 150 km
Run smoke  : True | Run full downloads: False


---
## 2. Interactive map — browse and download stations

The widget loads all Meteostat stations within the search radius. Click any marker to
open a popup with station info and a **Download** button. Use the rectangle tool to
select multiple stations and bulk-download them.

In [ ]:
if RUN_DOWNLOADS:
    MeteostatDownloader(
        zoom=7,
        center=(LAT, LON),
        radius=RADIUS_M,
        output_folder=OUTPUT_DIR,
    )
else:
    print('Interactive Meteostat widget skipped during scripted release execution.')

Interactive Meteostat widget skipped during scripted release execution.


---
## 3. Nearest stations — programmatic search

`get_meteostat_stations_nearby()` queries the Meteostat station catalogue and returns
all stations within `RADIUS_M` metres of the target point, sorted by distance.

Columns in the result:
- `name` — station name
- `country` — ISO country code
- `latitude`, `longitude`, `elevation` — station location
- `distance` — distance from target point (metres)

> **Tip:** A larger radius (e.g., 200 km) finds more candidates for scoring,
> but distant stations may have different climatology than your study area.
> For mountainous catchments, prefer stations at **similar elevation** over the nearest one.


In [ ]:
try:
    nearby = get_meteostat_stations_nearby(LAT, LON, radius=RADIUS_M, limit=SEARCH_LIMIT)
    print(f'{len(nearby)} stations within {RADIUS_M/1000:.0f} km of ({LAT}, {LON}):')
    if not nearby.empty:
        display(nearby[['name','country','latitude','longitude','elevation','distance']]
                .assign(dist_km=nearby['distance'] / 1000)
                .drop(columns='distance')
                .sort_values('dist_km').head(15))
except Exception as e:
    print(f'⚠ Meteostat station search failed: {e}')
    nearby = pd.DataFrame()

5 stations within 150 km of (43.259, -4.053):
                      name country  latitude  longitude  elevation   dist_km
id                                                                          
08021  Santander / Parayas      ES   43.4333    -3.8167          6   27.2167
08023            Santander      ES   43.4667    -3.8167         64   29.9716
08025     Bilbao / Sondica      ES   43.3000    -2.9333         42   90.7558
08075   Burgos / Villafria      ES   42.3667    -3.6333        894  104.9590
08080              Vitoria      ES   42.8833    -2.7167        513  116.3057


---
## 4. Score stations by data coverage

`score_meteostat_stations()` downloads a short sample period for each candidate station
and computes a **composite score** based on:
- % of non-null values per variable (`prcp_pct`, `temp_pct`, …)
- Distance penalty (closer stations score slightly higher)

The station with the highest `score` is the best choice for the requested variables.

> ⚠️ The score is computed on the **smoke period** (1 month by default), not the full record.
> A station that scores 100% in January 2020 may have gaps in older years.
> Always check `quality_df` after downloading the full series.


In [ ]:
try:
    if nearby.empty:
        print('No nearby stations available for scoring.')
    else:
        ranking = score_meteostat_stations(
            nearby,
            start_date=DOWNLOAD_START_DATE,
            end_date=DOWNLOAD_END_DATE,
            frequency=FREQUENCY,
            variables=['prcp', 'temp', 'tmin', 'tmax', 'rhum', 'wspd', 'pres'],
        )
        if not ranking.empty:
            cols = ['station_id', 'name', 'dist_km', 'prcp_pct', 'temp_pct', 'tmin_pct', 'tmax_pct', 'rhum_pct', 'score']
            display(ranking[[c for c in cols if c in ranking.columns]].head(10))
        else:
            print('No ranking data available.')
except Exception as e:
    print(f'⚠  {e}')

  station_id                 name   prcp_pct  ...  tmax_pct  rhum_pct       score
0      08025     Bilbao / Sondica  96.774194  ...     100.0     100.0  687.698614
1      08023            Santander  87.096774  ...     100.0     100.0  684.099614
2      08075   Burgos / Villafria  87.096774  ...     100.0     100.0  676.600874
3      08080              Vitoria  87.096774  ...     100.0     100.0  675.466204
4      08021  Santander / Parayas  74.193548  ...     100.0     100.0  671.471878

[5 rows x 8 columns]


---
## 5. Download — single station

Download daily data for a specific Meteostat station ID.
Meteostat station IDs match SYNOP WMO codes (e.g. `08021` = Santander/Parayas).

In [ ]:
df = download_meteostat_series(
    station_id   = STATION_ID,
    start_date   = DOWNLOAD_START_DATE,
    end_date     = DOWNLOAD_END_DATE,
    frequency    = FREQUENCY,
    station_name = STATION_NAME,
)

if df is not None and not df.empty:
    print(f'Station : {STATION_NAME} ({STATION_ID})')
    print(f'Period  : {df["date"].min()} → {df["date"].max()}')
    print(f'Rows    : {len(df)}')
    display(df.head())

    out_path = Path(OUTPUT_DIR) / f'series_{FREQUENCY}_{STATION_ID}.csv'
    df.to_csv(out_path, index=False)
    print(f'\nSaved → {out_path}')
else:
    print('⚠  No data returned for this station/period.')

Station : Santander / Parayas (08021)
Period  : 2020-01-01 → 2020-01-31
Rows    : 31
         date  tmed_celsius  ...  station_id         station_name
0  2020-01-01           9.1  ...       08021  Santander / Parayas
1  2020-01-02           7.2  ...       08021  Santander / Parayas
2  2020-01-03          10.4  ...       08021  Santander / Parayas
3  2020-01-04          10.4  ...       08021  Santander / Parayas
4  2020-01-05           8.0  ...       08021  Santander / Parayas

[5 rows x 16 columns]

Saved → /workspace/data/meteostat/series_daily_08021.csv


---
## 6. Auto-select best station near a point

`download_best_meteostat_series()` combines search + scoring + download in one call:
1. Searches all stations within `radius` metres
2. Scores them on the requested `variables`
3. Downloads the full period from the top-scoring station

Use this when you don't have a preferred station ID and want the best available data automatically.

```python
# Typical call for a hydrological study
df_best, best_id, ranking = download_best_meteostat_series(
    lat=LAT, lon=LON,
    start_date='1970-01-01', end_date='2024-12-31',
    radius=150_000,           # 150 km
    frequency='daily',
    variables=['prcp', 'tmin', 'tmax'],
)
```


In [ ]:
try:
    df_best, best_id, ranking_best = download_best_meteostat_series(
        lat        = LAT,
        lon        = LON,
        start_date = DOWNLOAD_START_DATE,
        end_date   = DOWNLOAD_END_DATE,
        radius     = RADIUS_M,
        frequency  = FREQUENCY,
        variables  = ['prcp', 'temp', 'tmin', 'tmax'],
    )
    if best_id and df_best is not None:
        print(f'Best station: {best_id}  ({len(df_best)} rows)')
        display(df_best.head())
    else:
        print('⚠  No suitable station found in radius.')
except Exception as e:
    print(f'⚠  {e}')

Best station: 08025  (31 rows)
         date  tmed_celsius  ...  station_id      station_name
0  2020-01-01           7.3  ...       08025  Bilbao / Sondica
1  2020-01-02           6.7  ...       08025  Bilbao / Sondica
2  2020-01-03           6.9  ...       08025  Bilbao / Sondica
3  2020-01-04           9.0  ...       08025  Bilbao / Sondica
4  2020-01-05           8.2  ...       08025  Bilbao / Sondica

[5 rows x 16 columns]


---
## 7. Quality analysis — downloaded data

`analyze_series_quality()` checks coverage and value ranges for all downloaded series.

| Column | Meaning | Threshold for concern |
|--------|---------|----------------------|
| `missing_percent` | % of days with NaN | >5%: infill; >10%: exclude from extremes |
| `full_years` | Years with <1% missing | Need ≥10 for reliable frequency analysis |
| `min_value` / `max_value` | Range check | Negative precip or >500 mm/day → data error |

> **Note on `precipitation_mm` vs `prcp`:** pyhydra renames Meteostat's `prcp` column
> to `precipitation_mm` for consistency with AEMET and OGIMET loaders.

After quality control, these series connect directly to:
- **Spatial interpolation** → `interpolation` notebook
- **Extreme value analysis** → `extreme_value_analysis` notebook
- **Stochastic generation** → `stochastic_generation` notebook


In [ ]:
loader = MeteostatCSVLoader(OUTPUT_DIR)

try:
    series_df = loader.load_series_data('precipitation_mm', frequency='daily')
    quality_df = loader.analyze_series_quality()
    display(quality_df)
except ValueError as e:
    print(f'⚠  {e}\n   Download data first in cell above.')
    quality_df = None

                station_id  start_year  ...  min_value  max_value
0  08021_Santander_Parayas        1970  ...        0.0      134.4
1                    08021        1970  ...        0.0       10.5

[2 rows x 10 columns]


---
## 8. Visualisation — time series + annual cycle

In [ ]:
try:
    wmo_result = find_station_by_wmo('08021')
    if not wmo_result.empty:
        display(wmo_result)
    else:
        print('Station not found in Meteostat catalog')
except Exception as e:
    print(f'⚠ Meteostat WMO lookup failed: {e}')

      id country region  latitude  longitude  elevation       timezone
0  08021      ES      S   43.4333    -3.8167          6  Europe/Madrid


---
## 9. OGIMET vs Meteostat — availability comparison

Quick summary of why Meteostat is the preferred source for routine downloads.

In [ ]:
comparison = pd.DataFrame({
    'Feature'       : ['Speed (55 years)', 'Rate limiting', 'IP blocks', 'API stability', 'Variables', 'Resolution'],
    'OGIMET'        : ['~11 min+', 'Yes (1s sleep/chunk)', 'Yes (after heavy use)', 'Scraping HTML (fragile)', 'Full SYNOP', 'Sub-daily (raw SYNOP)'],
    'Meteostat'     : ['< 30 s', 'No', 'No', 'Official Python API', 'Daily + hourly', 'Daily / Hourly'],
}).set_index('Feature')

display(comparison)
print('\nRecommendation: use Meteostat for long-period daily/hourly downloads.')
print('Use OGIMET only when sub-daily raw SYNOP messages are required.')

                                   OGIMET            Meteostat
Feature                                                       
Speed (55 years)                 ~11 min+               < 30 s
Rate limiting        Yes (1s sleep/chunk)                   No
IP blocks           Yes (after heavy use)                   No
API stability     Scraping HTML (fragile)  Official Python API
Variables                      Full SYNOP       Daily + hourly
Resolution          Sub-daily (raw SYNOP)       Daily / Hourly

Recommendation: use Meteostat for long-period daily/hourly downloads.
Use OGIMET only when sub-daily raw SYNOP messages are required.


---
## 10. Find station by WMO code

Both OGIMET and Meteostat use WMO station codes (e.g. `08021`). Use `find_station_by_wmo()`
to cross-reference them.

In [ ]:
wmo_result = find_station_by_wmo('08021')
if not wmo_result.empty:
    display(wmo_result)
else:
    print('Station not found in Meteostat catalog')

      id country region  latitude  longitude  elevation       timezone
0  08021      ES      S   43.4333    -3.8167          6  Europe/Madrid


---
## ⚠️ Known limitations and data gaps

**Coverage by region:**
- North America, Europe, Japan: excellent coverage (many long-record stations)
- Africa, South America, Central Asia: sparse (many stations with <10 years of data)
- Station density decreases before 1950 — check `start_year` in quality output

**Variable completeness:**
- Precipitation (`prcp`) is the most consistently available variable
- Humidity (`rhum`), sunshine (`tsun`), and snow (`snow`) have frequent gaps
- Hourly data is available for far fewer stations than daily

**Data quality issues to watch for:**
- Sudden jumps in series (station relocation, instrument change) — not automatically flagged
- Duplicated days in some old records
- `max_value` > 200 mm/day is not necessarily wrong (extreme events) but warrants manual check

**Infilling gaps:**
```python
# Simple: forward-fill short gaps (≤3 days)
df_filled = df['precipitation_mm'].fillna(method='ffill', limit=3)

# Better: spatial interpolation from neighbouring stations
# → see interpolation notebook
```
